# 🎲 Sprint 0 — Stochastic Thinking & Monte Carlo

Companion ao **MIT 6.0002 (Fall 2016)**, aulas 4-10.

**Como trabalhar:**
- Cada secção corresponde a uma aula do Guttag.
- Cada `# TODO` é uma micro-task. Substitui a expressão `...` ou completa o esqueleto.
- A célula corre uma `assert` — se falhar, lê a mensagem e corrige.
- **NÃO edites este ficheiro** (`scaffolded.ipynb`). Trabalha em `solutions/local.ipynb` (cópia local).

**Regra de ouro:** se uma assertion passar à primeira sem pensares, provavelmente saltaste algo. Reflete no porquê.

## ⚙️ Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Reprodutibilidade — fixa a seed quando estiveres a debugar.
# Tira esta linha quando quiseres ver variabilidade real.
rng = np.random.default_rng(seed=42)

# Atalho para asserção com mensagem visual.
def check(cond, msg='falhou'):
    assert cond, f'❌ {msg}'
    print('✅', msg)

## §1 — Random Walks

**Aulas Guttag:** L4 (Stochastic Thinking), L5 (Random Walks).

Um *random walk* é a coisa mais simples que ainda contém todo o esqueleto da estatística inferencial: amostragem, expectativa, variância, leis de grande números. Vamos construí-lo do zero.

### Task 1.1 — Random Walk 1D

Implementa `walk_1d(n_steps)` que devolve a **posição final** depois de `n_steps` passos, cada passo +1 ou −1 com probabilidade 1/2.

Usa `rng.choice([-1, 1], size=n_steps)` e soma.

In [ ]:
def walk_1d(n_steps):
    """Devolve a posição final após n_steps passos uniformes em {-1, +1}."""
    # TODO: gera os passos com rng.choice e devolve a soma
    return ...

# Sanity checks
pos = walk_1d(0)
check(pos == 0, 'walk_1d(0) deve devolver 0 (zero passos = não saiu da origem)')

# Para n grande, posição típica deve ser O(sqrt(n)) — testa que não explode
samples = np.array([walk_1d(1000) for _ in range(2000)])
check(abs(samples.mean()) < 5, f'média ~0 esperada, obteve {samples.mean():.2f}')
check(20 < samples.std() < 50, f'std ~sqrt(1000)≈31, obteve {samples.std():.2f}')

### Task 1.2 — Distância média escala como √n

Para `n_steps` em `[10, 100, 1000, 10000, 100000]`, simula `n_trials=2000` walks e calcula `E[|posição final|]`.

Plota `E[|x|]` vs `√n_steps` em log-log. Devias ver uma reta de declive ~1.

**Pergunta:** porque é que NÃO escala com `n` mas sim com `√n`?

In [ ]:
def expected_abs_distance(n_steps, n_trials=2000):
    """Estima E[|posição_final|] simulando n_trials walks."""
    # TODO: corre n_trials walks de n_steps passos e devolve a média de |posição|
    return ...

ns = np.array([10, 100, 1000, 10000, 100000])
expected_abs = np.array([expected_abs_distance(n) for n in ns])

# Em teoria E[|x_n|] ≈ sqrt(2n/π) ≈ 0.798 sqrt(n)
ratio = expected_abs / np.sqrt(ns)
check(np.all(np.abs(ratio - 0.798) < 0.1), 
      f'E[|x|]/sqrt(n) deve estabilizar perto de 0.798, obteve {ratio}')

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(ns, expected_abs, 'o-', label='E[|x|] simulado')
ax.loglog(ns, 0.798 * np.sqrt(ns), '--', label='0.798·√n (teoria)')
ax.set_xlabel('n (passos)'); ax.set_ylabel('E[|posição|]')
ax.legend(); ax.set_title('Random walk: distância escala como √n')
plt.show()

### Task 1.3 — Random Walk 2D

Implementa `walk_2d(n_steps)` que devolve **a trajetória completa** (`shape=(n_steps+1, 2)`, começa em `(0,0)`).

Cada passo escolhe uma de 4 direções (N/S/E/W) com igual probabilidade.

Plota uma trajetória de 10 000 passos. Vais ver fractal-like clusters — não é coincidência.

In [ ]:
def walk_2d(n_steps):
    """Trajetória 2D, shape (n_steps+1, 2), começa em (0, 0)."""
    # Escolhe um índice em {0,1,2,3} para cada passo, mapeia para vetor unitário
    directions = np.array([[0, 1], [0, -1], [1, 0], [-1, 0]])
    # TODO: gera n_steps índices, indexa em 'directions', acumula com cumsum
    # TODO: prepende a origem (0,0) para a trajetória ter shape (n_steps+1, 2)
    return ...

trail = walk_2d(10_000)
check(trail.shape == (10_001, 2), f'shape deve ser (10001, 2), obteve {trail.shape}')
check(np.all(trail[0] == 0), 'trajetória deve começar em (0, 0)')

# diferenças consecutivas devem ter norma 1 (passo unitário)
diffs = np.diff(trail, axis=0)
norms = np.linalg.norm(diffs, axis=1)
check(np.allclose(norms, 1.0), f'cada passo deve ter norma 1, obteve {norms[:5]}')

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(trail[:, 0], trail[:, 1], lw=0.5)
ax.plot(0, 0, 'go', label='início'); ax.plot(*trail[-1], 'ro', label='fim')
ax.set_aspect('equal'); ax.legend(); ax.set_title('Random walk 2D, 10k passos')
plt.show()

### Task 1.4 — Walk enviesado

Modifica para um passo `+1` com probabilidade `p` e `-1` com probabilidade `1-p`.

Para `p=0.55, n=1000, n_trials=5000`, qual a fração de walks que termina ABAIXO de zero?

**Intuição esperada:** com bias positivo de 0.05, a fração abaixo de zero deve ser muito pequena para n=1000.

**Conexão:** este é exatamente o argumento que distingue "vantagem estatística pequena" de "ruído". Volta aqui quando estudares apostas e finanças.

In [ ]:
def biased_walk_final(n_steps, p, n_trials):
    """Devolve array de tamanho n_trials com a posição final de cada walk."""
    # TODO: simula n_trials walks de n_steps com Bernoulli(p)·2 - 1 como passo
    return ...

finals = biased_walk_final(1000, p=0.55, n_trials=5000)
frac_below_zero = float(np.mean(finals < 0))
print(f'fração abaixo de zero: {frac_below_zero:.4f}')

# A teoria diz que com p=0.55, drift = 1000·(2p-1) = 100, std ≈ sqrt(1000) ≈ 31.6.
# Ou seja, posição final é ~N(100, 31.6) e P(X<0) é minúscula (~10^-4).
check(frac_below_zero < 0.01, 
      f'fração esperada < 1%; obteve {frac_below_zero:.4f}')

# E a média deve estar perto do drift teórico
check(abs(finals.mean() - 100) < 5, 
      f'média esperada ~100 (drift), obteve {finals.mean():.1f}')

## §2 — Monte Carlo Simulation

**Aula Guttag:** L6.

Monte Carlo = substituir uma integral analítica por uma média empírica de samples. Funciona em **qualquer** dimensão (esta é a ideia que abre as portas para MCMC, RL, e métodos estocásticos em finanças).

### Task 2.1 — Estimar π por dart-throwing

Atira `n` pontos uniformemente em `[-1,1]² `. A fração que cai dentro do círculo unitário aproxima π/4.

Implementa `estimate_pi(n)`.

In [ ]:
def estimate_pi(n):
    """Estima π com n samples uniformes em [-1,1]^2."""
    # TODO: gera n pontos 2D em [-1,1]^2 com rng.uniform
    # TODO: calcula a fração com norma <= 1
    # TODO: devolve essa fração * 4
    return ...

for n in [100, 10_000, 1_000_000]:
    est = estimate_pi(n)
    print(f'n={n:>7}: π ≈ {est:.5f} (erro {abs(est - np.pi):.5f})')

# Para n=1M, erro absoluto deve ser <0.01 com alta probabilidade
check(abs(estimate_pi(1_000_000) - np.pi) < 0.01,
      'estimativa com 1M samples deve estar a <0.01 de π')

### Task 2.2 — Convergência: erro vs √n

Para `n` em `[100, 1_000, 10_000, 100_000, 1_000_000]`, repete a estimativa `30` vezes e calcula o erro absoluto **médio**.

Plota erro_médio vs n em log-log. Esperas declive ≈ −0.5 (Monte Carlo converge como `1/√n`).

**Implicação:** para uma casa decimal extra de precisão, precisas de **100×** mais samples.

In [ ]:
def mean_pi_error(n, n_repeats=30):
    # TODO: corre estimate_pi n_repeats vezes e devolve a média de |est - π|
    return ...

ns = np.array([100, 1_000, 10_000, 100_000, 1_000_000])
errs = np.array([mean_pi_error(n) for n in ns])

# Ajustar declive em log-log com np.polyfit
slope, intercept = np.polyfit(np.log(ns), np.log(errs), 1)
print(f'declive log-log = {slope:.3f} (esperado ≈ -0.5)')
check(-0.65 < slope < -0.35, f'declive deve estar perto de -0.5, obteve {slope:.3f}')

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(ns, errs, 'o-', label='erro empírico')
ax.loglog(ns, 1/np.sqrt(ns), '--', label='1/√n (referência)')
ax.set_xlabel('n samples'); ax.set_ylabel('erro absoluto médio')
ax.legend(); ax.set_title('Monte Carlo converge como 1/√n')
plt.show()

### Task 2.3 — Integral via Monte Carlo

Estima `∫₀¹ x² dx` (resposta exacta = 1/3) com Monte Carlo: amostra `n` pontos uniformes em `[0,1]`, avalia `x²` em cada um, devolve a média.

Esta é a forma mais simples do *Law of the Unconscious Statistician*: `E[g(X)] ≈ (1/n) Σ g(x_i)` quando os `x_i` são i.i.d. da distribuição de `X`.

In [ ]:
def estimate_integral_x_squared(n):
    # TODO: amostra n uniformes em [0,1], calcula x^2, devolve a média
    return ...

for n in [100, 10_000, 1_000_000]:
    est = estimate_integral_x_squared(n)
    print(f'n={n:>7}: ∫x²dx ≈ {est:.5f} (verdade=0.33333, erro {abs(est - 1/3):.5f})')

check(abs(estimate_integral_x_squared(1_000_000) - 1/3) < 0.005,
      'estimativa com 1M samples deve estar a <0.005 da verdade')

## §3 — Confidence Intervals

**Aula Guttag:** L7.

Quando dizes "a média é 12 ± 1.5 com 95% confiança", o que isso significa? **Não** é "P(verdade ∈ [10.5, 13.5]) = 0.95". É: "se eu repetir esta experiência 100 vezes, ~95 dos intervalos que construir desta forma vão conter a verdade". Subtileza importante.

### Task 3.1 — CI normal-aproximada para a média

Para uma amostra `x` de tamanho `n`, com média amostral `m` e desvio padrão amostral `s`:

$$ \text{CI}_{95\%} = m \pm 1.96 \cdot \frac{s}{\sqrt{n}} $$

(Onde `1.96` é o quantil 0.975 da Normal padrão.) Implementa `normal_ci_95(x)`.

In [ ]:
def normal_ci_95(x):
    """Devolve (low, high), o IC 95% normal-aproximado para a média."""
    x = np.asarray(x)
    n = len(x)
    # TODO: calcula a média amostral
    # TODO: calcula o desvio padrão amostral (com ddof=1, divisão por n-1)
    # TODO: calcula o erro padrão SE = s / sqrt(n)
    # TODO: devolve (m - 1.96·SE, m + 1.96·SE)
    return ...

# Teste com amostra Normal(5, 2), n=200
sample = rng.normal(loc=5, scale=2, size=200)
low, high = normal_ci_95(sample)
print(f'IC 95% = [{low:.3f}, {high:.3f}], média amostral = {sample.mean():.3f}')
check(low < sample.mean() < high, 'a média amostral deve estar dentro do seu próprio IC')
check(0.1 < high - low < 1.0, f'largura razoável esperada, obteve {high - low:.3f}')

### Task 3.2 — Validar a cobertura via simulação

Aqui está a versão concreta do que "95% confiança" significa: simula 5000 experiências; em cada uma, gera amostra de tamanho 100 de `Normal(0, 1)`, calcula o IC, e regista se contém a média verdadeira (`0`). A fração de aciertos deve ficar perto de 0.95.

**Pergunta:** se aumentares o `n_per_experiment`, a cobertura muda? Porquê / porque não?

In [ ]:
def coverage_simulation(n_per_experiment=100, n_experiments=5000, true_mean=0.0, true_sd=1.0):
    """Devolve a fração de ICs que contêm a true_mean."""
    contains = 0
    for _ in range(n_experiments):
        sample = rng.normal(loc=true_mean, scale=true_sd, size=n_per_experiment)
        low, high = normal_ci_95(sample)
        # TODO: incrementa contains se true_mean estiver dentro de [low, high]
        ...
    return contains / n_experiments

cov = coverage_simulation()
print(f'cobertura empírica do IC 95%: {cov:.3f} (esperado ≈ 0.95)')
check(0.93 < cov < 0.97, f'cobertura deve estar em [0.93, 0.97], obteve {cov:.3f}')

## §4 — Bootstrap

**Aula Guttag:** L8.

O bootstrap é mágico: dá-te um intervalo de confiança para **qualquer estatística** (mediana, percentil 90, qualquer função da amostra) sem teoria fechada. A receita: re-amostra com reposição da tua amostra original `B` vezes, calcula a estatística em cada réplica, e usa a distribuição empírica como proxy para a distribuição amostral verdadeira.

### Task 4.1 — Bootstrap CI para a média

Implementa `bootstrap_ci_mean(x, n_resamples=2000, conf=0.95)` que:

1. Reamostra `x` com reposição `n_resamples` vezes, cada réplica do mesmo tamanho que `x`.
2. Calcula a média de cada réplica.
3. Devolve os percentis `(1-conf)/2` e `1-(1-conf)/2` dessa distribuição.

Compara com a CI normal-aproximada que fizeste em §3. Para amostras Normais grandes, devem coincidir.

In [ ]:
def bootstrap_ci_mean(x, n_resamples=2000, conf=0.95):
    """IC bootstrap percentil para a média."""
    x = np.asarray(x)
    n = len(x)
    means = np.empty(n_resamples)
    for i in range(n_resamples):
        # TODO: amostra n índices em [0, n) com reposição (rng.integers)
        # TODO: calcula a média do reamostre x[idx] e guarda em means[i]
        ...
    alpha = (1 - conf) / 2
    # TODO: devolve (np.quantile(means, alpha), np.quantile(means, 1 - alpha))
    return ...

sample = rng.normal(loc=5, scale=2, size=200)
boot_lo, boot_hi = bootstrap_ci_mean(sample)
norm_lo, norm_hi = normal_ci_95(sample)
print(f'bootstrap: [{boot_lo:.3f}, {boot_hi:.3f}]')
print(f'normal:    [{norm_lo:.3f}, {norm_hi:.3f}]')

# As duas larguras devem coincidir a ~10% para amostras grandes Normais
check(abs((boot_hi - boot_lo) - (norm_hi - norm_lo)) < 0.1,
      'bootstrap e normal devem dar ICs de largura comparável para Normais grandes')

### Task 4.2 — Bootstrap para a mediana (onde a normal não funciona)

A mediana não tem fórmula fechada simples para a CI. O bootstrap é a abordagem padrão.

Generaliza para `bootstrap_ci_stat(x, stat_fn, n_resamples, conf)`: aceita qualquer função `stat_fn` (e.g., `np.median`, `np.std`).

In [ ]:
def bootstrap_ci_stat(x, stat_fn, n_resamples=2000, conf=0.95):
    """IC bootstrap percentil para uma estatística arbitrária."""
    x = np.asarray(x)
    n = len(x)
    stats = np.empty(n_resamples)
    for i in range(n_resamples):
        # TODO: reamostra com reposição e aplica stat_fn
        ...
    alpha = (1 - conf) / 2
    # TODO: devolve quantis (alpha, 1-alpha)
    return ...

# Distribuição assimétrica (lognormal) para ver a diferença média vs mediana
asym = rng.lognormal(mean=0.0, sigma=0.5, size=300)
med_lo, med_hi = bootstrap_ci_stat(asym, np.median)
mean_lo, mean_hi = bootstrap_ci_stat(asym, np.mean)
print(f'mediana IC95%: [{med_lo:.3f}, {med_hi:.3f}], mediana amostral = {np.median(asym):.3f}')
print(f'média IC95%:   [{mean_lo:.3f}, {mean_hi:.3f}], média amostral   = {asym.mean():.3f}')

# A média amostral é >> mediana amostral em distribuições com cauda direita
check(asym.mean() > np.median(asym),
      'em lognormal, média deve ser > mediana — cauda à direita')

### Task 4.3 — Quando o bootstrap mente?

O bootstrap NÃO é mágico universal. Falha em estatísticas perto da fronteira do suporte (e.g., o **máximo**). Vê com os teus olhos:

Gera amostra `Uniform(0, θ)` com `θ=1`. O *MLE* do máximo é `max(x)`. Aplica bootstrap para o máximo. Vê o que acontece.

In [ ]:
uniform_sample = rng.uniform(0, 1, size=100)
max_lo, max_hi = bootstrap_ci_stat(uniform_sample, np.max)
print(f'IC95% para o máximo: [{max_lo:.4f}, {max_hi:.4f}]')
print(f'máximo amostral    : {uniform_sample.max():.4f}')

# Observa: o limite superior do IC é literalmente o max amostral.
# O bootstrap não vê acima do que a amostra já viu, mas o θ verdadeiro é > max(x).
# Resultado: IC é fortemente enviesado para baixo. Lição: estatísticas de extremos
# precisam de teoria especializada (e.g., extreme value theory).
check(max_hi <= uniform_sample.max() + 1e-9,
      'limite superior do IC bootstrap nunca excede o máximo amostral — daí o viés')

## §5 — Two-sample t-test

**Aula Guttag:** L10.

Tens duas amostras (e.g., grupo de controlo vs grupo de tratamento). Pergunta: as médias são significativamente diferentes? O *Welch's t-test* responde sem assumir variâncias iguais.

### Task 5.1 — Welch's t-statistic from scratch

$$ t = \frac{\bar{x}_1 - \bar{x}_2}{\sqrt{s_1^2/n_1 + s_2^2/n_2}} $$

Onde `s²` é a variância amostral (com `ddof=1`). Implementa `welch_t(x1, x2)` que devolve o valor de `t`.

In [ ]:
def welch_t(x1, x2):
    """Welch's t-statistic, sem scipy."""
    x1 = np.asarray(x1); x2 = np.asarray(x2)
    # TODO: m1, m2 = médias
    # TODO: v1, v2 = variâncias com ddof=1
    # TODO: n1, n2 = tamanhos
    # TODO: devolve (m1 - m2) / sqrt(v1/n1 + v2/n2)
    return ...

a = rng.normal(0, 1, 50)
b = rng.normal(0.5, 1, 50)
t_stat = welch_t(a, b)
print(f't-statistic ≈ {t_stat:.3f}')
check(t_stat < 0, 'média de a < média de b → t deve ser negativo')
check(abs(t_stat) > 1.5, f't deve ser "grande" (>1.5) com n=50 e diff de 0.5σ, obteve {t_stat:.3f}')

### Task 5.2 — p-value via simulação (permutation test)

Sem scipy: estima o p-value por **permutação**. Sob a hipótese nula "as duas amostras vêm da mesma distribuição", as etiquetas (grupo 1 vs grupo 2) são intercambiáveis. Logo:

1. Junta `x1` e `x2` num só array `pool`.
2. Permuta `pool` aleatoriamente, atribui os primeiros `n1` ao grupo 1 e o resto ao grupo 2.
3. Calcula `t` permutado. Repete `B=2000` vezes.
4. p-value = fração de `|t_perm| ≥ |t_observado|`.

In [ ]:
def permutation_pvalue(x1, x2, n_perm=2000):
    x1 = np.asarray(x1); x2 = np.asarray(x2)
    n1 = len(x1)
    pool = np.concatenate([x1, x2])
    t_obs = abs(welch_t(x1, x2))
    extreme = 0
    for _ in range(n_perm):
        # TODO: rng.permutation(pool); divide em (primeiros n1) e (resto); calcula t
        # TODO: incrementa extreme se |t_perm| >= t_obs
        ...
    return extreme / n_perm

# Caso 1: distribuições mesmo iguais — esperar p ~ uniforme (não próximo de 0)
p_null = permutation_pvalue(rng.normal(0, 1, 100), rng.normal(0, 1, 100))
print(f'p-value sob nula verdadeira: {p_null:.3f}')
check(p_null > 0.05, f'sob nula verdadeira, p deve ser "normal" (>0.05 quase sempre), obteve {p_null:.3f}')

# Caso 2: diferença real grande — p deve ser ~0
p_alt = permutation_pvalue(rng.normal(0, 1, 100), rng.normal(1.0, 1, 100))
print(f'p-value com diferença real (Δμ=1σ): {p_alt:.3f}')
check(p_alt < 0.01, f'com diferença grande, p deve ser <0.01, obteve {p_alt:.3f}')

### Task 5.3 — Power analysis empírica

*Power* = probabilidade de **rejeitar** a nula quando ela é falsa. Tipicamente queres `power ≥ 0.8`.

Para `effect_size` em `[0.0, 0.2, 0.5, 0.8, 1.0]` (diferença de médias em unidades de σ) e `n=30` por grupo, simula `n_experiments=500` t-tests e estima a fracção que rejeita ao nível `α=0.05`.

**Resultado esperado:** com `effect=0`, power ≈ α = 0.05 (taxa de erro tipo I). Para `effect=0.8`, power deve estar perto de 0.5-0.6 com n=30.

In [ ]:
def power_simulation(effect_size, n=30, n_experiments=500, alpha=0.05):
    """Fracção de t-tests que rejeitam H0 quando a diferença real é effect_size."""
    rejects = 0
    for _ in range(n_experiments):
        x1 = rng.normal(0, 1, n)
        x2 = rng.normal(effect_size, 1, n)
        # TODO: usa permutation_pvalue (com n_perm baixo, e.g. 200, para acelerar)
        # TODO: incrementa rejects se p < alpha
        ...
    return rejects / n_experiments

for eff in [0.0, 0.2, 0.5, 0.8, 1.0]:
    pwr = power_simulation(eff)
    print(f'effect={eff:.1f}σ → power={pwr:.3f}')

# Sob nula verdadeira, power deve ≈ alpha
check(power_simulation(0.0) < 0.10,
      'sob H0 verdadeira, taxa de rejeição deve estar perto de α=0.05')
# Para effect grande, power claramente acima de alpha
check(power_simulation(1.0) > 0.5,
      'com efeito de 1σ e n=30, power deve estar acima de 0.5')

## 🏁 Sprint 0 — fechado

Se chegaste aqui sem `AssertionError`, dominaste:

1. **Random walks** — esqueleto de toda a estatística inferencial.
2. **Monte Carlo** — substituir integrais por médias, com convergência `1/√n` quantificada.
3. **CI normal** — e o que "95% confiança" *realmente* quer dizer (cobertura empírica).
4. **Bootstrap** — e onde ele falha (estatísticas de extremos).
5. **Permutation test** — p-values sem assumir Normal, e power analysis empírica.

**Próximo sprint:** [Sprint 1 — Convex Optimization](../Sprint01_Optimization/). A regressão linear vai sair como caso particular de gradient descent. A intuição estatística construída aqui passa a ser combustível para análise de convergência.